### Config

In [5]:
import os
from google.cloud import aiplatform
from dotenv import load_dotenv
from transformers import pipeline

load_dotenv() 

PROJECT_ID = os.environ["PROJECT_ID"]
LOCATION = os.environ["LOCATION"]

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"./service_account.json"

aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
)

### Model

In [6]:
classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [7]:
result = classifier("I love this product")

In [8]:
result

[{'label': 'POSITIVE', 'score': 0.9998788833618164}]

### For Checking

In [14]:
models = aiplatform.Model.list()

for model in models:
    print("Name:", model.display_name)
    print("Resource:", model.resource_name)
    print("----")

Name: huggingface-model
Resource: projects/344969539300/locations/us-central1/models/3135542180114857984
----


In [15]:
endpoints = aiplatform.Endpoint.list()

for ep in endpoints:
    print("Name:", ep.display_name)
    print("Resource:", ep.resource_name)
    print("----")

Name: huggingface-endpoint
Resource: projects/344969539300/locations/us-central1/endpoints/7288021015492296704
----


### Uploade Model and Create Endpoint

In [4]:
DOCKER_IMAGE_NAME="us-docker.pkg.dev/deeplearning-platform-release/gcr.io/huggingface-pytorch-inference-cu121.2-2.transformers.4-44.ubuntu2204.py311"

In [9]:
HF_MODEL_ID = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
HF_TASK = "text-classification"

In [10]:
model = aiplatform.Model.upload(
    display_name="huggingface-model",
    serving_container_image_uri=DOCKER_IMAGE_NAME,
    serving_container_environment_variables={
        "HF_MODEL_ID": HF_MODEL_ID,
        "HF_TASK": HF_TASK,
    },
)
model.wait()

Creating Model
Create Model backing LRO: projects/344969539300/locations/us-central1/models/3135542180114857984/operations/3763661983010783232
Model created. Resource name: projects/344969539300/locations/us-central1/models/3135542180114857984@1
To use this Model in another session:
model = aiplatform.Model('projects/344969539300/locations/us-central1/models/3135542180114857984@1')


In [13]:
endpoint = aiplatform.Endpoint.create(
    display_name="huggingface-endpoint"
)

print("Endpoint created:")
print(endpoint.resource_name)

Creating Endpoint
Create Endpoint backing LRO: projects/344969539300/locations/us-central1/endpoints/7288021015492296704/operations/8281335359216812032
Endpoint created. Resource name: projects/344969539300/locations/us-central1/endpoints/7288021015492296704
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/344969539300/locations/us-central1/endpoints/7288021015492296704')
Endpoint created:
projects/344969539300/locations/us-central1/endpoints/7288021015492296704


### Deploy Endpoint

In [16]:
model = aiplatform.Model(
    "projects/344969539300/locations/us-central1/models/3135542180114857984"
)

In [17]:
model.deploy(
    endpoint=endpoint,
    deployed_model_display_name="huggingface-model-deployed",
    machine_type="g2-standard-12",
    accelerator_type="NVIDIA_L4",
    accelerator_count=1,
    min_replica_count=1,
    max_replica_count=1,
)

Deploying model to Endpoint : projects/344969539300/locations/us-central1/endpoints/7288021015492296704
Deploy Endpoint model backing LRO: projects/344969539300/locations/us-central1/endpoints/7288021015492296704/operations/1461759623471038464
Endpoint model deployed. Resource name: projects/344969539300/locations/us-central1/endpoints/7288021015492296704


resource name: projects/344969539300/locations/us-central1/endpoints/7288021015492296704

### Inference

In [18]:
endpoint = aiplatform.Endpoint(
    "projects/344969539300/locations/us-central1/endpoints/7288021015492296704"
)

In [20]:
instances=["I love this product", "I hate this product"]
response = endpoint.predict(instances=instances, parameters={"top_k": 2})

In [21]:
response

Prediction(predictions=[[{'score': 0.9998788833618164, 'label': 'POSITIVE'}, {'score': 0.0001210561968036927, 'label': 'NEGATIVE'}], [{'score': 0.9997544884681702, 'label': 'NEGATIVE'}, {'score': 0.0002454846107866615, 'label': 'POSITIVE'}]], deployed_model_id='1038484884143734784', metadata=None, model_version_id='1', model_resource_name='projects/344969539300/locations/us-central1/models/3135542180114857984', explanations=None)

### Clean

In [22]:
# Clean
endpoint = aiplatform.Endpoint(
    "projects/344969539300/locations/us-central1/endpoints/7288021015492296704"
)

endpoint.delete(force=True)

model = aiplatform.Model(
    "projects/344969539300/locations/us-central1/models/3135542180114857984"
)

model.delete()

Undeploying Endpoint model: projects/344969539300/locations/us-central1/endpoints/7288021015492296704
Undeploy Endpoint model backing LRO: projects/344969539300/locations/us-central1/endpoints/7288021015492296704/operations/4598094551469522944
Endpoint model undeployed. Resource name: projects/344969539300/locations/us-central1/endpoints/7288021015492296704
Deleting Endpoint : projects/344969539300/locations/us-central1/endpoints/7288021015492296704
Endpoint deleted. . Resource name: projects/344969539300/locations/us-central1/endpoints/7288021015492296704
Deleting Endpoint resource: projects/344969539300/locations/us-central1/endpoints/7288021015492296704
Delete Endpoint backing LRO: projects/344969539300/locations/us-central1/operations/8273031847403847680
Endpoint resource projects/344969539300/locations/us-central1/endpoints/7288021015492296704 deleted.
Deleting Model : projects/344969539300/locations/us-central1/models/3135542180114857984
Model deleted. . Resource name: projects/3